# Solutions · 01 · Phase equilibrium and vapour-liquid equilibrium

Worked solutions to the exercises of [notebook 01](../notebooks/01_phase_equilibrium_and_vle.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engsep import datasets, vle
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Fit a one-parameter Margules model (ln gamma_1 = A x_2^2, ln gamma_2 = A x_1^2) to the ethanol-water data by least squares on y and compare its x-y curve and azeotrope with Wilson's. Where does the simpler model fail?

In [2]:
d = pd.read_csv(datasets.path("vle_ethanol_water.csv"), comment="#")
pair = ("ethanol", "water")
def y_margules(A, x1):
    out = []
    for x in x1:
        def f(T):
            g1, g2 = np.exp(A * (1 - x) ** 2), np.exp(A * x ** 2)
            return x * g1 * vle.antoine_pressure(pair[0], T) + (1 - x) * g2 * vle.antoine_pressure(pair[1], T) - 101325.0
        T = optimize.brentq(f, 300, 400)
        out.append(x * np.exp(A * (1 - x) ** 2) * vle.antoine_pressure(pair[0], T) / 101325.0)
    return np.array(out)
xs, ys = d.x_ethanol.to_numpy()[1:-1], d.y_ethanol.to_numpy()[1:-1]
A_fit = optimize.minimize_scalar(lambda A: np.sum((y_margules(A, xs) - ys) ** 2), bounds=(0, 3), method="bounded").x
err_m = np.abs(y_margules(A_fit, xs) - ys)
err_w = np.abs(np.array([vle.bubble_y(pair, x)[1] for x in xs]) - ys)
az_m = optimize.brentq(lambda x: y_margules(A_fit, [x])[0] - x, 0.5, 0.99)
print(f"one-parameter Margules: A = {A_fit:.3f}; mean |dy| = {err_m.mean():.4f} (Wilson {err_w.mean():.4f}); azeotrope at x = {az_m:.3f} (Wilson {vle.azeotrope(*pair)['x']:.3f}, data 0.894)")
print("largest Margules errors at x =", np.round(xs[np.argsort(err_m)[-3:]], 3))

one-parameter Margules: A = 1.308; mean |dy| = 0.0310 (Wilson 0.0064); azeotrope at x = 0.818 (Wilson 0.885, data 0.894)
largest Margules errors at x = [0.261 0.396 0.327]


A symmetric one-parameter model cannot fit an asymmetric system: ethanol-water has $\gamma^\infty$ of about 6
for ethanol in water but under 3 for water in ethanol, and Margules with one $A$ splits the difference - its
errors are five times Wilson's, largest in the middle of the range where the real curve bends, and its
azeotrope lands at 0.82 instead of 0.89. Two parameters are the minimum for an asymmetric binary.

## Exercise 2

For acetone-methanol (a minimum-boiling azeotrope), compute the azeotrope composition at 0.5, 1, 2 and 5 atm and explain how pressure-swing distillation could separate the pair.

In [3]:
for P_atm in (0.5, 1.0, 2.0, 5.0):
    az = vle.azeotrope("acetone", "methanol", P=P_atm * 101325.0)
    print(f"{P_atm:.1f} atm: acetone-methanol azeotrope at x = {az['x']:.3f}, {az['T']-273.15:.1f} degC" if az else f"{P_atm:.1f} atm: no azeotrope")

0.5 atm: acetone-methanol azeotrope at x = 0.893, 37.0 degC
1.0 atm: acetone-methanol azeotrope at x = 0.794, 55.4 degC
2.0 atm: acetone-methanol azeotrope at x = 0.676, 76.1 degC
5.0 atm: acetone-methanol azeotrope at x = 0.494, 107.4 degC


The azeotrope moves from acetone-rich at low pressure towards methanol-rich at high pressure. Two columns at
different pressures can therefore each overshoot the other's azeotrope: the low-pressure column takes the
mixture past the high-pressure azeotrope composition, the high-pressure column past the low-pressure one, and
the recycled azeotropic streams shrink. Pressure-swing distillation works when the azeotrope moves by several
percent per doubling of pressure, as here.

## Exercise 3

**Implement it yourself:** write the Wilson activity-coefficient equations from the Lambda parameters and reproduce `vle.wilson_gamma` for ethanol-water at x = 0.2 and 350 K.

In [4]:
x1, T = 0.2, 350.0
L12, L21 = vle.wilson_lambda(("ethanol", "water"), T)
x2 = 1 - x1
term = L12 / (x1 + L12 * x2) - L21 / (x2 + L21 * x1)
g1 = np.exp(-np.log(x1 + L12 * x2) + x2 * term)
g2 = np.exp(-np.log(x2 + L21 * x1) - x1 * term)
print(f"by hand: gamma = ({g1:.4f}, {g2:.4f}); library: {tuple(np.round(vle.wilson_gamma(('ethanol', 'water'), x1, T), 4))}")

by hand: gamma = (2.2019, 1.1027); library: (np.float64(2.2019), np.float64(1.1027))


The Wilson equations in six lines; the library evaluates the same expressions with the same Lambda
parameters.